# Demo: Sahel Tree Cover Change Analysis

This notebook demonstrates the core statistical analysis and figure generation from:

> **Farmer Managed Natural Regeneration Promotes Expansion of Trees on Croplands in the Sahel** — *Nature Sustainability* (under review)

It runs entirely on `significant_trends_ls789_wc_hct.csv`, which is committed in this repository and contains per-tile (0.1° grid) tree cover change statistics aggregated from the full Landsat 7/8/9 analysis (2000–2020) across the Sahel.

**Expected run time on a normal desktop computer: < 2 minutes**

**Expected output:** A bar chart showing net tree cover change by Sahel country. Niger stands out with positive net change (tree cover gain), consistent with the paper's main finding on FMNR.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

print('pandas:', pd.__version__)
print('matplotlib:', matplotlib.__version__)
print('numpy:', np.__version__)

## 1. Load the included demo data

Columns:
- `tile_id`: 0.1° grid tile identifier
- `country`: ISO3 country code
- `region`: sub-national region
- `pos_sig_diff`: mean area of statistically significant tree cover **gain** per pixel (m²)
- `neg_sig_diff`: mean area of statistically significant tree cover **loss** per pixel (m²)

In [ ]:
df = pd.read_csv('significant_trends_ls789_wc_hct.csv')
print(f'Total tiles: {len(df):,}  |  Countries: {df["country"].nunique()}')
df[['tile_id', 'country', 'region', 'pos_sig_diff', 'neg_sig_diff']].head()

## 2. Compute net tree cover change per country

In [ ]:
country_rename = {
    'GMB': 'Gambia', 'BFA': 'Burkina Faso', 'MLI': 'Mali',
    'NER': 'Niger', 'NGA': 'Nigeria', 'SEN': 'Senegal',
    'TCD': 'Chad', 'ETH': 'Ethiopia', 'SDN': 'Sudan',
    'CMR': 'Cameroon', 'MRT': 'Mauritania', 'GIN': 'Guinea',
    'CIV': "Côte d'Ivoire", 'GHA': 'Ghana', 'BEN': 'Benin',
    'TGO': 'Togo', 'ERI': 'Eritrea', 'SSD': 'South Sudan',
    'CAF': 'Central African Republic', 'DJI': 'Djibouti',
}

df['country_name'] = df['country'].map(country_rename).fillna(df['country'])

# Net change = gain - loss (mean per tile, in m² of tree cover per pixel)
df['net_change'] = df['pos_sig_diff'] - df['neg_sig_diff']

# Aggregate to country level
by_country = (
    df.groupby('country_name')[['pos_sig_diff', 'neg_sig_diff', 'net_change']]
    .mean()
    .sort_values('net_change', ascending=False)
)

print(by_country[['pos_sig_diff', 'neg_sig_diff', 'net_change']]
      .multiply(1e6).round(1)  # convert to µm² for readability
      .rename(columns=lambda c: c + ' (×10⁻⁶ m²/px)'))

## 3. Plot: net tree cover change by country

Niger shows a positive net change — consistent with widespread FMNR adoption on Sahelian croplands. Most other countries show net tree cover loss over the same period.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

colors = ['#2e7d32' if v > 0 else '#c62828' for v in by_country['net_change']]
ax.bar(by_country.index, by_country['net_change'] * 1e6,
       color=colors, alpha=0.85, width=0.6)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(0.5)
ax.spines['left'].set_linewidth(0.5)

ax.set_ylabel('Net tree cover change\n(×10⁻⁶ m² per pixel)', fontsize=12)
ax.set_xlabel('Country', fontsize=12)
ax.set_title('Statistically significant tree cover change 2000–2020\n(green = net gain, red = net loss)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.tight_layout()
plt.savefig('demo_output_net_change_by_country.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to demo_output_net_change_by_country.png')

## 4. Summary statistics

In [ ]:
n_gain = (by_country['net_change'] > 0).sum()
n_loss = (by_country['net_change'] <= 0).sum()
print(f'Countries with net tree cover GAIN: {n_gain}')
print(f'Countries with net tree cover LOSS: {n_loss}')

if 'Niger' in by_country.index:
    niger = by_country.loc['Niger']
    print(f'\nNiger:')
    print(f'  Gain: {niger["pos_sig_diff"]*1e6:.2f} ×10⁻⁶ m²/px')
    print(f'  Loss: {niger["neg_sig_diff"]*1e6:.2f} ×10⁻⁶ m²/px')
    print(f'  Net:  {niger["net_change"]*1e6:.2f} ×10⁻⁶ m²/px (positive = tree cover expansion)')